<a href="https://colab.research.google.com/github/mukul-mschauhan/GenerativeAI/blob/main/Introduction_to_Agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Introduction to AI Agents
### Powered by OpenAI GPT-4o-mini + LangGraph + Tavily

---

## 🎯 What You Will Learn

This notebook takes you on a **progressive journey** from a simple LLM call all the way to a fully autonomous **Market Intelligence Agent** with web search capabilities and a chat UI.

| Stage | What We Build | Concept Introduced |
|-------|--------------|--------------------|
| 1 | Plain LLM Call | GenAI Assistant (NOT an agent) |
| 2 | LLM + One Custom Tool | Tool Calling |
| 3 | LLM + Web Search (Tavily) | Real-World Grounding |
| 4 | Full Market Analyst Agent | ReAct Loop + Memory + System Prompt |
| 5 | Gradio Chat UI | Productionizing the Agent |

---

## 🧠 Key Concept: What Makes Something an "Agent"?

```
┌─────────────────────────────────────────────────────────┐
│                   AI AGENT ANATOMY                      │
│                                                         │
│   User Input                                            │
│       │                                                 │
│       ▼                                                 │
│  ┌─────────┐    decides    ┌──────────┐                 │
│  │   LLM   │ ────────────▶ │  Tools   │  (web, code,   │
│  │ (Brain) │ ◀──────────── │ (Hands)  │   APIs, etc.)  │
│  └─────────┘   observes    └──────────┘                 │
│       │                                                 │
│       ▼                                                 │
│   Final Answer                                          │
│                                                         │
│   🔁 This loop is called the ReAct (Reason + Act) loop  │
└─────────────────────────────────────────────────────────┘
```

> **A plain LLM = GenAI Assistant** → It only generates text from memory.  
> **LLM + Tools + Loop = AI Agent** → It can take actions in the real world.

## 📦 Step 0 — Install Dependencies

We install the full stack needed:

| Package | Role |
|---------|------|
| `openai` | Official OpenAI Python SDK |
| `langchain-openai` | LangChain wrapper for OpenAI models |
| `langgraph` | Agent orchestration framework (the "engine") |
| `langchain-core` | Core LangChain abstractions (tools, messages, prompts) |
| `langchain-community` | Community integrations (Tavily search tool wrapper) |
| `langchain-tavily` | Official Tavily integration for LangChain |
| `tavily-python` | Tavily web search client |
| `gradio` | Chat UI framework |
| `python-dotenv` | Load API keys from `.env` file |

In [1]:
!pip install -qU openai langchain-openai langgraph langchain-core langchain-community
!pip install -qU langchain-tavily tavily-python gradio python-dotenv rich

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.7/502.7 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

## 🔑 Step 0b — Import Libraries & Configure API Keys

### Library Map

```python
openai                        # Core OpenAI SDK
langchain_openai.ChatOpenAI   # LangChain-compatible OpenAI chat model
langgraph.prebuilt            # Pre-built ReAct agent graph
langchain_core.tools          # @tool decorator for defining tools
langchain_core.messages       # HumanMessage, AIMessage, SystemMessage
```

### API Keys Required
- **`OPENAI_API_KEY`** → [platform.openai.com/api-keys](https://platform.openai.com/api-keys)
- **`TAVILY_API_KEY`** → [tavily.com](https://tavily.com) (free tier available — 1000 searches/month)

> 💡 **Colab Tip:** Use `google.colab.userdata` (Secrets tab 🔑 in sidebar) instead of hardcoding keys.

In [2]:
import os
import warnings
warnings.filterwarnings("ignore")

from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

# ── Load API Keys ──────────────────────────────────────────────────────────────
# Option A: Google Colab Secrets (recommended for Colab users)
try:
    from google.colab import userdata
    OPENAI_API_KEY  = userdata.get("OPENAI_API_KEY")
    TAVILY_API_KEY  = userdata.get("TAVILY_API_KEY")
    print("✅ API keys loaded from Colab Secrets.")

except Exception:
    # Option B: Environment variables / .env file (for local Jupyter)
    from dotenv import load_dotenv
    load_dotenv()
    OPENAI_API_KEY  = os.getenv("OPENAI_API_KEY")
    TAVILY_API_KEY  = os.getenv("TAVILY_API_KEY")
    print("✅ API keys loaded from environment variables.")

# Set keys in environment (required by LangChain/Tavily wrappers)
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

print(f"OpenAI key ending in: ...{OPENAI_API_KEY[-4:]}")
print(f"Tavily key ending in:  ...{TAVILY_API_KEY[-4:]}")

✅ API keys loaded from Colab Secrets.
OpenAI key ending in: ...TjgA
Tavily key ending in:  ...FFcX


---
# 🧪 Step 1 — Plain LLM Call (This is NOT an Agent)

Before building an agent, we must understand what an agent is **not**.

A plain LLM call:
- Takes a text prompt → Returns text
- Has **no tools**, **no loop**, **no memory**
- Cannot check the weather, run code, browse the web, or do anything beyond generating text
- This is a **GenAI Assistant**, not an Agent

```
User ──[prompt]──▶ LLM ──[text]──▶ User
         (one-shot, no actions taken)
```

We use the **OpenAI Python SDK** directly here (not LangChain), to show the raw API pattern.

In [3]:
# ── Direct OpenAI SDK call (raw, no LangChain) ────────────────────────────────
client = OpenAI(api_key=OPENAI_API_KEY)

response = client.chat.completions.create(
    model="gpt-4o-mini",                        # Fast, cheap, capable model
    messages=[
        {"role": "user",
         "content": "In one sentence, explain what an AI agent is."}
    ]
)

print(response.choices[0].message.content.strip())

An AI agent is a software entity that autonomously perceives its environment, makes decisions, and takes actions to achieve specific goals based on its programming and learned experiences.


### 📌 Observation

The LLM gave us a definition — but it **did not verify it**, **did not search the web**, and **cannot take any action**. It only knows what was baked into its training weights.

> ⚠️ **This is a GenAI Assistant, not an AI Agent.**

To build an agent, we need to give the LLM **tools** — functions it can call to interact with the outside world. That's what comes next.

---
# 🛠️ Step 2 — Add One Custom Tool (Now It Becomes an Agent!)

A **tool** is any Python function decorated with `@tool`. The key things:
- The **docstring** tells the LLM what the tool does and when to use it
- The **type hints** on parameters tell the LLM what input to pass
- The LLM autonomously **decides** whether and when to call it

We start with a **dummy weather tool** that always returns "sunny". The point is NOT the weather data — it's to demonstrate the **tool-calling mechanism** in isolation.

```
User: "What is the weather in Delhi?"
         │
         ▼
       LLM thinks: "I need weather info → I should call get_weather(city='Delhi')"
         │
         ▼
   get_weather("Delhi") → "The weather in Delhi is sunny."
         │
         ▼
       LLM: "The weather in Delhi is sunny."
```

In [4]:
# ── Define a Custom Tool with @tool decorator ─────────────────────────────────
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city.
    Use this whenever the user asks about weather in any location.
    """
    # Dummy response for teaching purposes
    return f"The weather in {city} is sunny and 28°C."


# Inspect what the tool looks like to the LLM
print("Tool name:", get_weather.name)
print("Tool description:", get_weather.description)

Tool name: get_weather
Tool description: Get the current weather for a given city.
    Use this whenever the user asks about weather in any location.


### 🏗️ Building the Agent with LangGraph's `create_react_agent`

We now wire together:
1. **`ChatOpenAI`** — the OpenAI LLM (brain)
2. **`[get_weather]`** — list of tools (hands)
3. **`create_react_agent`** — the orchestration loop (engine)

`create_react_agent` implements the **ReAct** pattern:
- **Re**ason → LLM decides what to do
- **Act** → calls a tool
- **Observe** → reads the tool result
- Loops until a final answer is ready

In [5]:
# ── Initialise ChatOpenAI (LangChain-compatible OpenAI wrapper) ───────────────
llm = ChatOpenAI(
    model="gpt-4o-mini",        # Recommended: fast + affordable for demos
    api_key=OPENAI_API_KEY,
    temperature=0               # 0 = deterministic, best for agents
)

# ── Create the ReAct Agent ────────────────────────────────────────────────────
agent_basic_tool = create_react_agent(
    model=llm,
    tools=[get_weather]
)

# ── Run the Agent ─────────────────────────────────────────────────────────────
result = agent_basic_tool.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in Delhi?"}]}
)

# The last message in the conversation is the agent's final answer
print("🤖 Agent Answer:", result["messages"][-1].content)

🤖 Agent Answer: The weather in Delhi is sunny with a temperature of 28°C.


### 🔍 (Optional) Inspect the Full Reasoning Trace

You can inspect every message in the agent's internal conversation to see the full **Reason → Act → Observe** cycle. This is invaluable for debugging.

In [6]:
# ── Print the full message trace ──────────────────────────────────────────────
print("\n📋 Full Agent Message Trace:")
print("=" * 50)
for i, msg in enumerate(result["messages"]):
    role = type(msg).__name__
    content = msg.content if msg.content else "[Tool call / Tool result]"
    print(f"[{i}] {role}: {content[:200]}")
print("=" * 50)


📋 Full Agent Message Trace:
[0] HumanMessage: What is the weather in Delhi?
[1] AIMessage: [Tool call / Tool result]
[2] ToolMessage: The weather in Delhi is sunny and 28°C.
[3] AIMessage: The weather in Delhi is sunny with a temperature of 28°C.


---
# 🌐 Step 3 — Add Tavily: A Real Web Search Tool

The dummy weather tool proved the mechanics. Now we give the agent **real internet access** using **Tavily** — a search engine optimized specifically for LLM agents.

### Why Tavily over Google/Bing?

| Feature | Tavily | Google Search API |
|---------|--------|------------------|
| LLM-optimized output | ✅ Clean text | ❌ Raw HTML / noisy |
| Free tier | ✅ 1000 req/month | ❌ Paid |
| LangChain native | ✅ First-class | ⚠️ Wrapper needed |
| Answer synthesis | ✅ Built-in | ❌ Manual |

With Tavily, the agent can answer questions that require **real-time information** beyond GPT's training cutoff.

In [7]:
from langchain_tavily import TavilySearch

# ── Create the Tavily search tool ─────────────────────────────────────────────
tavily_tool = TavilySearch(
    max_results=3,          # Return top-3 web results per query
    search_depth="advanced" # "advanced" gives better quality than "basic"
)

# ── Wire Tavily into a fresh ReAct agent ──────────────────────────────────────
agent_search = create_react_agent(
    model=llm,
    tools=[tavily_tool]
)

# ── Ask a question that requires current, real-world information ──────────────
query = "Tell me one fun fact about AI Agents that most people don't know."
result = agent_search.invoke(
    {"messages": [{"role": "user", "content": query}]}
)

print("🌐 Agent Answer (with live web search):")
print(result["messages"][-1].content)

🌐 Agent Answer (with live web search):
One fun fact about AI agents that many people might not know is that they can exhibit "emergent behavior." This means that when multiple AI agents interact in a shared environment, they can develop complex behaviors that were not explicitly programmed into them. For example, in simulations, AI agents can form groups, develop strategies, or even create their own languages to communicate, showcasing a level of adaptability and intelligence that can surprise researchers and developers alike. This phenomenon highlights the potential for AI systems to learn and evolve in ways that are not fully predictable.


---
# 🏢 Step 4 — Full Market Intelligence Agent

## The Problem Statement

A financial analyst or sales professional needs to quickly profile a new prospect or competitor. Traditionally, they would:
1. Open multiple tabs
2. Search for the company
3. Read recent news articles
4. Synthesize findings
5. Write a structured summary

⏱️ This takes **30–45 minutes per company**.

## The Agentic Solution

We build an AI Agent that does all of this **autonomously in under 60 seconds**, using the **four pillars of agency**:

```
┌──────────────────────────────────────────────────────────┐
│               MARKET INTELLIGENCE AGENT                  │
│                                                          │
│  🧠 PLANNING      → Breaks goal into sub-tasks           │
│                     (core business → news → outlook)     │
│                                                          │
│  ⚡ DECISION      → Decides to use Tavily Search         │
│                     to fetch real-time data              │
│                                                          │
│  🔬 REASONING     → Reads results, synthesizes           │
│                     into structured markdown report      │
│                                                          │
│  💾 MEMORY        → Remembers conversation history       │
│                     for follow-up questions              │
└──────────────────────────────────────────────────────────┘
```

## 🛠️ Step 4a — Tools: The Agent's Hands

An LLM alone is a **brain** — brilliant at reasoning, planning, and synthesis.  
But without tools, it's locked inside its training data.

We equip our agent with **Tavily Search** — giving it real-time internet access.  
The agent will **autonomously decide** when to search and what to search for.

In [8]:
from langchain_tavily import TavilySearch

# ── Configure the search tool ─────────────────────────────────────────────────
# max_results=3: Agent gets 3 web results per search query.
# The agent may search MULTIPLE times (e.g., once for core business, once for news)
search_tool = TavilySearch(max_results=3)

tools = [search_tool]

print("✅ Tools registered:")
for t in tools:
    print(f"  • {t.name}: {t.description[:80]}...")

✅ Tools registered:
  • tavily_search: A search engine optimized for comprehensive, accurate, and trusted results. Usef...


## 🧠 Step 4b — The Brain: GPT-4o-mini as our LLM

### Model Choice: `gpt-4o-mini`

| Model | Speed | Cost | Quality | Best For |
|-------|-------|------|---------|----------|
| `gpt-4o-mini` | ⚡ Fast | 💰 Low | ⭐⭐⭐⭐ | Demos, agents, teaching |
| `gpt-4o` | Medium | Medium | ⭐⭐⭐⭐⭐ | Production, complex tasks |
| `o1-mini` | Slow | High | ⭐⭐⭐⭐⭐ | Deep reasoning tasks |

> `gpt-4o-mini` is the **recommended model for this notebook** — it's fast, cheap, and fully capable of running the ReAct loop reliably.

### System Prompt: The Agent's Persona

The system prompt is the **blueprint for the agent's behaviour**. It defines:
- **Role** → Who the agent is
- **Goal** → What it needs to accomplish
- **Plan** → How it should approach tasks step-by-step
- **Guardrails** → What it must NOT do (no hallucination)

In [9]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage

# ── The LLM (Brain) ───────────────────────────────────────────────────────────
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
    temperature=0               # Deterministic output for consistent reports
)

# ── System Prompt (The Agent's Persona + Planning Instructions) ───────────────
SYSTEM_PROMPT = """You are an elite Market Intelligence Agent.
Your mission is to deliver concise, accurate, and actionable company analyses.

When asked about a company, ALWAYS follow this structured plan:

STEP 1 — Core Business Search:
  Search the web to find out exactly what the company does (products, services, business model).

STEP 2 — Recent News Search:
  Search the web for the company's most recent news, funding, partnerships, or strategic moves.

STEP 3 — Synthesize & Report:
  Compile your findings into a clean, structured markdown report with the following sections:
  ## 🏢 Core Business
  ## 📰 Recent News & Developments
  ## 🎯 Strategic Outlook

RULES:
- Always search before answering. Do NOT rely on training data alone.
- Do not hallucinate. If you don't know, search.
- Be concise. Executives are busy. Keep each section to 3–5 bullet points.
"""

print("✅ LLM and System Prompt configured.")
print(f"   Model: {llm.model_name}")
print(f"   Temperature: {llm.temperature}")

✅ LLM and System Prompt configured.
   Model: gpt-4o-mini
   Temperature: 0.0


## 💾 Step 4c — Memory: The Agent's Short-Term Notepad

Without memory, every message starts a fresh conversation. The agent would forget context between turns.

With `MemorySaver`, the agent remembers the **entire conversation history** within a session (identified by `thread_id`). This enables natural follow-up questions:

```
You: "Analyze Anthropic"
Agent: [searches + reports on Anthropic]

You: "Who are their main competitors?"   ← agent remembers "Anthropic" from above
Agent: [searches for Anthropic competitors]

You: "Compare their safety approach"     ← still in context
Agent: [answers in context]
```

> 📝 `thread_id` is like a session ID. Each unique `thread_id` maintains its own separate conversation history.

In [10]:
from langgraph.checkpoint.memory import MemorySaver

# ── In-memory checkpointer (persists conversation within this runtime) ─────────
memory = MemorySaver()

# ── Session config — change thread_id to start a fresh conversation ────────────
config = {"configurable": {"thread_id": "market_analyst_session_1"}}

print("✅ Memory (MemorySaver) initialised.")
print(f"   Session thread_id: {config['configurable']['thread_id']}")

✅ Memory (MemorySaver) initialised.
   Session thread_id: market_analyst_session_1


## ⚙️ Step 4d — Assemble the Full Agent

We wire together:
- **LLM** (brain)
- **Tools** (hands)
- **System Prompt** (persona + planning)
- **Memory** (conversation history)

The `create_react_agent` function from LangGraph builds the complete **stateful agent graph** — a compiled workflow that handles the full ReAct reasoning loop automatically.

In [11]:
from langgraph.prebuilt import create_react_agent

# ── Assemble the complete Market Intelligence Agent ───────────────────────────
agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,       # System prompt passed as string (LangGraph 0.4+)
    checkpointer=memory         # Enables conversation memory across turns
)

print("✅ Market Intelligence Agent assembled.")
print("   Components: LLM + Tavily Search + System Prompt + MemorySaver")

✅ Market Intelligence Agent assembled.
   Components: LLM + Tavily Search + System Prompt + MemorySaver


## 🚀 Step 4e — Run the Agent: Analyze a Company

Let's put the agent to work. It will:
1. Receive our query
2. Autonomously decide to search the web (multiple times)
3. Read and synthesize the results
4. Output a structured markdown report

> ⏱️ Expect this to take 10–30 seconds — the agent is making multiple live web searches.

In [13]:
# ── First query: Analyze a company ───────────────────────────────────────────
result = agent.invoke(
    {"messages": [HumanMessage(content="Analyze the Current Situation of the Indian Stock Market (NSE)")]},
    config=config
)

print(result["messages"][-1].content)

## 🏢 Current Situation of the Indian Stock Market (NSE)
- **Market Performance**: The Nifty 50 index is currently trading below 24,600, with the Sensex experiencing a decline of over 500 points. This reflects a risk-off sentiment among investors amid geopolitical tensions, particularly related to the US-Iran situation.
- **Sector Performance**: The banking and auto sectors are underperforming, while the IT sector has shown some resilience. The India VIX, a measure of market volatility, has surged, indicating increased uncertainty.
- **Foreign Investment Trends**: Foreign Institutional Investors (FIIs) have reportedly withdrawn approximately ₹17,000 crore from IT stocks in February, driven by concerns over AI and market conditions.
- **Market Sentiment**: Analysts suggest that while the current market conditions are challenging, there is no immediate need for panic. However, caution is advised as earnings projections may be affected by ongoing geopolitical issues.
- **Broader Market Tre

### 🔁 Follow-Up Question (Memory in Action)

Because `MemorySaver` is active, we can now ask a **follow-up question without repeating context**. The agent remembers the full conversation.

In [ ]:
# ── Follow-up query (agent uses memory from the previous turn) ────────────────
followup = agent.invoke(
    {"messages": [HumanMessage(content="Who are their main competitors and how do they compare?")]},
    config=config  # Same thread_id = same conversation)

print(followup["messages"][-1].content)

---
# 🖥️ Step 5 — Gradio Chat UI: Productionize the Agent

The agent works perfectly via code. Now let's wrap it in a **chat interface** using Gradio so it looks and feels like a real product — a polished chat window, right inside Colab.

### What Gradio Does
- Provides a **web-based chat UI** with conversation history
- Manages the `message / history` loop automatically
- Auto-generates a **public share URL** (valid for 72 hours) for Colab demos

> 💡 The `gr.ChatInterface` pattern is the simplest way to expose any Python function as a chat app. Our `chat_with_agent()` function acts as the bridge between Gradio's UI and our LangGraph agent.

In [ ]:
import gradio as gr

# ── Bridge function: Gradio UI ↔ LangGraph Agent ──────────────────────────────
def chat_with_agent(message: str, history: list) -> str:
    """
    Called by Gradio on every user message.
    - message: the current user input string
    - history: list of [user, assistant] pairs (Gradio manages this)
    Returns the agent's response as a string.
    """
    # Use a fixed thread_id so all messages in this UI session share memory
    session_config = {"configurable": {"thread_id": "gradio_ui_session"}}

    result = agent.invoke(
        {"messages": [HumanMessage(content=message)]},
        config=session_config
    )

    return result["messages"][-1].content


# ── Build the Gradio Chat Interface ──────────────────────────────────────────
demo = gr.ChatInterface(
    fn=chat_with_agent,
    title="📈 Market Intelligence Agent",
    description=(
        "Powered by **OpenAI GPT-4o-mini** + **Tavily Web Search** + **LangGraph ReAct**\n\n"
        "Ask me to analyze any company. I will search the web, gather recent news, "
        "and generate a structured strategic report."
    ),
    examples=[
        "Analyze Tata Motors",
        "What are the latest developments in the Indian stock market?,
        "How do we see Indian Financial Markets during Iran Israel War"])

# ── Launch ────────────────────────────────────────────────────────────────────
# share=True: Creates a public URL (auto-enabled in Colab)
# debug=True: Shows logs and errors inline
demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2376e51d32a38542f0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
# 📚 Summary & Key Takeaways

## What We Built

| Component | Technology | Role |
|-----------|-----------|------|
| LLM (Brain) | `ChatOpenAI` / `gpt-4o-mini` | Reasoning, planning, synthesis |
| Tool (Hands) | `TavilySearch` | Real-time web search |
| Agent Loop | `create_react_agent` (LangGraph) | ReAct orchestration |
| Memory | `MemorySaver` | Conversation persistence |
| Persona | System Prompt | Behaviour + planning instructions |
| UI | `gr.ChatInterface` | Chat frontend |

## The Agent Lifecycle

```
User Input
    │
    ▼
┌─────────────────────────────────────┐
│         ReAct Loop (LangGraph)      │
│                                     │
│  LLM Thinks → Should I use a tool? │
│       │                             │
│    YES │                 NO         │
│       ▼                  ▼          │
│  Call Tool          Final Answer   │
│  (Tavily)                           │
│       │                             │
│  Observe Result                     │
│       │                             │
│  Loop back to Think                 │
└─────────────────────────────────────┘
```

## What's Next?

- 🔧 **Add more tools** → code execution, database queries, email sending
- 🧠 **Add long-term memory** → vector store for persistent knowledge
- 🤝 **Multi-agent systems** → multiple specialised agents collaborating
- 🛡️ **Add guardrails** → input/output validation with Pydantic
---
